## Import

In [18]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd

from data import load_and_split
from evaluate import evaluate
from compare import per_user_metrics, paired_difference
from models.popularity import Popularity
from models.random_model import RandomModel
from models.pure_svd import PureSVD
from models.ials import IALS
from models.bpr import BPR
from models.ease import EASE

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
train, val, test = load_and_split('../data/ml-1m/ratings.dat')
print(train['user'].nunique(), train['item'].nunique())
print(len(train), len(val), len(test))

6034 3125
562308 6034 6034


## Random model and popularity model

In [3]:
models = {
    'random': RandomModel(),
    'popularity': Popularity(),
}

rows = []
for name, model in models.items():
    model.fit(train)
    result = evaluate(model, train, val, k=10)
    rows.append({'model': name, **result})

results = pd.DataFrame(rows).set_index('model')
results

,recall,ndcg
model,,
random,0.003315,0.001480
popularity,0.045907,0.021854


## Pure SVD

In [5]:
for d in [8, 16, 32, 64, 128]:
    model = PureSVD(n_factors=d).fit(train)
    print(d, evaluate(model, train, val, k=10))

8 {'recall': 0.06861120318196884, 'ndcg': 0.03488220697320986}
16 {'recall': 0.08468677494199536, 'ndcg': 0.04070826661105542}
32 {'recall': 0.0962877030162413, 'ndcg': 0.04794390835681603}
64 {'recall': 0.09695061319191249, 'ndcg': 0.04812864903196527}
128 {'recall': 0.09065296652303613, 'ndcg': 0.04401016771465734}


#### Лучший ранг на val - 32-64, далее при d -> ранг матрицы модель переобучается

## IALS

In [10]:
for alpha in [1, 10, 40]:
    for reg in [0.01, 0.1, 1.0]:
        model = IALS(n_factors=64, alpha=alpha, reg=reg, n_iter=15).fit(train)
        print(alpha, reg, evaluate(model, train, val, k=10))

1 0.01 {'recall': 0.10407689758037786, 'ndcg': 0.051536303075763006}
1 0.1 {'recall': 0.10357971494862446, 'ndcg': 0.051244882813730235}
1 1.0 {'recall': 0.10341398740470666, 'ndcg': 0.05118868523850439}
10 0.01 {'recall': 0.0956247928405701, 'ndcg': 0.046833001958514774}
10 0.1 {'recall': 0.0952933377527345, 'ndcg': 0.04695585864646066}
10 1.0 {'recall': 0.0952933377527345, 'ndcg': 0.046575545968425766}
40 0.01 {'recall': 0.08352668213457076, 'ndcg': 0.03975394802372333}
40 0.1 {'recall': 0.08319522704673517, 'ndcg': 0.04037962347770481}
40 1.0 {'recall': 0.08584686774941995, 'ndcg': 0.04143393594070581}


In [11]:
for alpha in [0.1, 0.3, 1, 3]:
    for reg in [0.1, 10, 100, 1000]:
        model = IALS(n_factors=64, alpha=alpha, reg=reg, n_iter=15).fit(train)
        print(alpha, reg, evaluate(model, train, val, k=10))

0.1 0.1 {'recall': 0.09910507126284389, 'ndcg': 0.04932079390410328}
0.1 10 {'recall': 0.10175671196552867, 'ndcg': 0.05034270787605182}
0.1 100 {'recall': 0.05104408352668213, 'ndcg': 0.02338931628917207}
0.1 1000 {'recall': 0.04342061650646337, 'ndcg': 0.019943868163969262}
0.3 0.1 {'recall': 0.10043089161418628, 'ndcg': 0.04959239199728495}
0.3 10 {'recall': 0.10076234670202187, 'ndcg': 0.05026583203801558}
0.3 100 {'recall': 0.058501823002983094, 'ndcg': 0.028244987535898478}
0.3 1000 {'recall': 0.04342061650646337, 'ndcg': 0.019943868163969262}
1 0.1 {'recall': 0.10357971494862446, 'ndcg': 0.051244882813730235}
1 10 {'recall': 0.10324825986078887, 'ndcg': 0.05118163683642535}
1 100 {'recall': 0.06165064633742128, 'ndcg': 0.031422889915676724}
1 1000 {'recall': 0.04342061650646337, 'ndcg': 0.019943868163969262}
3 0.1 {'recall': 0.09993370898243288, 'ndcg': 0.05051686894500978}
3 10 {'recall': 0.10159098442161087, 'ndcg': 0.050508101855856614}
3 100 {'recall': 0.08402386476632416, '

#### Большие а ухудшают результат. Большие лямбда эквивалентны сильному сжатию ранга и ведут к поведению, похожему на популярность

## BPR

In [14]:
for lr in [0.01, 0.05]:
    for n_iter in [50, 100, 200]:
        model = BPR(n_factors=64, learning_rate=lr, n_iter=n_iter).fit(train)
        print(lr, n_iter, evaluate(model, train, val, k=10))

0.01 50 {'recall': 0.0755717600265164, 'ndcg': 0.03665893558143433}
0.01 100 {'recall': 0.07756049055353, 'ndcg': 0.03825200800384719}
0.01 200 {'recall': 0.07341730195558502, 'ndcg': 0.036283686771892636}
0.05 50 {'recall': 0.07209148160424263, 'ndcg': 0.03505879122013302}
0.05 100 {'recall': 0.07043420616506463, 'ndcg': 0.034595997329535694}
0.05 200 {'recall': 0.07408021213125622, 'ndcg': 0.036560839779903105}


In [16]:
for reg in [10, 100, 500, 2000]:
    model = EASE(reg=reg).fit(train)
    print(reg, evaluate(model, train, val, k=10))

10 {'recall': 0.08866423599602254, 'ndcg': 0.045056392415203594}
100 {'recall': 0.10291680477295327, 'ndcg': 0.050687736823199464}
500 {'recall': 0.10623135565130924, 'ndcg': 0.051293521715953215}
2000 {'recall': 0.09910507126284389, 'ndcg': 0.04849960627460699}
